In [4]:
import tensorflow as tf
import keras_hub
# import tensorflow_datasets as tfds

In [ ]:
BERT_NAME = "bert_tiny_en_uncased"
SEQ_LEN = 512
NUM_CLASSES = 3
MODEL_PATH = "./2 LLMs/kaggle_sentiment/bert_tiny_en_uncased_sentiment.keras"

classifier = tf.keras.models.load_model(MODEL_PATH, compile=False)
classifier.summary()

In [18]:
imbd_ds = tf.data.Dataset.load("./2 LLMs/imdb_reviews/val_ds")
imbd_ds = imbd_ds.batch(64)
imbd_ds = imbd_ds.prefetch(tf.data.AUTOTUNE)

In [31]:
# Do not evaluate the loaded classifier directly on the IMDB dataset.
# It is still a 3-class Kaggle classifier. Use binary_classifier below.
classifier.evaluate(imbd_ds)

32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - loss: 1.3864 - sparse_categorical_accuracy: 0.5283


[1.3863611221313477, 0.5283203125]

### Reduce the 3-class classifier to 2 classes

In [29]:
# Kaggle label order: 0=neutral, 1=positive, 2=negative
# Binary IMDB label order: 0=negative, 1=positive
text_input = tf.keras.Input(shape=(), dtype=tf.string, name="text")
bert_inputs = classifier.preprocessor(text_input)
logits_3 = classifier(bert_inputs)
logits_2 = tf.keras.layers.Lambda(
        lambda logits: tf.gather(logits, [2, 1], axis=-1),
        name="negative_positive_logits",
        )(logits_3)

binary_classifier = tf.keras.Model(text_input, logits_2, name="binary_sentiment_from_3_class")
binary_classifier.compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["sparse_categorical_accuracy"],
        )
binary_classifier.summary()

Model: "binary_sentiment_from_3_class"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text (InputLayer)   │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bert_text_classifi… │ [(None, 512),     │          0 │ text[0][0]        │
│ (BertTextClassifie… │ (None, 512),      │            │                   │
│                     │ (None, 512)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bert_text_classifi… │ (None, 3)         │  4,386,307 │ bert_text_classi… │
│ (BertTextClassifie… │                   │            │ bert_text_classi… │
│                     │                   │            │ bert_text_classi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ negative_positive_… │ (None, 2)         │          0 │ bert_text_classi… │
│ (Lambda)            │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,386,307 (16.73 MB)

 Trainable params: 4,386,307 (16.73 MB)

 Non-trainable params: 0 (0.00 B)

In [30]:
binary_classifier.evaluate(imbd_ds)


32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - loss: 0.6711 - sparse_categorical_accuracy: 0.7002


[0.6711325645446777, 0.7001953125]